In [1]:
"""
Threshold robustness check for Reviewer 2, Comment 1.


"""

import argparse
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def build_out_degree(fr_matrix: pd.DataFrame, theta: float) -> pd.Series:
    """
    Reproduce the manuscript's network-construction convention:
    A_ij = 1 if FR_ij > theta, else 0.
    An edge is drawn j -> i (predictor influences target), so a
    company's "influence count" = its OUT-degree = the number of
    rows i for which that company (as column j) exceeds theta.
    """
    A = (fr_matrix > theta).astype(int)
    out_degree = A.sum(axis=0)   # sum down each column j
    return out_degree


def run_window(fr_matrix: pd.DataFrame, window_label: str):
    N = fr_matrix.shape[0]

    thresholds = {
        "mean (theta = 1/N, used in paper)": fr_matrix.values.mean(),
        "median": np.median(fr_matrix.values),
        "60th percentile": np.percentile(fr_matrix.values, 60),
        "75th percentile": np.percentile(fr_matrix.values, 75),
    }

    degree_by_threshold = {}
    print(f"\n{'='*70}\nWindow: {window_label}\n{'='*70}")
    for name, theta in thresholds.items():
        out_degree = build_out_degree(fr_matrix, theta)
        degree_by_threshold[name] = out_degree
        top5 = out_degree.sort_values(ascending=False).head(5)
        print(f"\n[{name}]  theta = {theta:.5f}")
        print(f"  mean out-degree: {out_degree.mean():.2f}")
        print(f"  top 5 by influence: {list(top5.index)}")

    # Spearman rank correlation of every alternative threshold vs. the
    # paper's mean-threshold (1/N) ranking
    base_name = "mean (theta = 1/N, used in paper)"
    base = degree_by_threshold[base_name]
    print(f"\nSpearman rank correlation vs. '{base_name}':")
    for name, series in degree_by_threshold.items():
        if name == base_name:
            continue
        rho, pval = spearmanr(base, series)
        print(f"  {name:<20}: rho = {rho:.3f}, p = {pval:.2e}")

    # Track specific firms of interest (edit this list as needed)
    watch_list = ["BP", "Gazprom", "Inter_Rao", "State_Grid", "ExxonMobil"]
    print(f"\nRank position (1 = most influential, out of {N}) for key firms:")
    for name, series in degree_by_threshold.items():
        ranks = series.rank(ascending=False)
        row = {}
        for firm in watch_list:
            match = [c for c in ranks.index if c.strip() == firm]
            if match:
                row[firm] = int(ranks[match[0]])
        print(f"  {name:<35}: {row}")

    return degree_by_threshold


def main():
    parser = argparse.ArgumentParser(description="Threshold robustness check")
    parser.add_argument("--window", type=int, default=None,
                         help="Window number 1-12. Omit to run all windows.")
    # parse_known_args (not parse_args) so this also runs cleanly inside
    # Jupyter/IPython, which injects its own "-f kernel-....json" argument
    # that argparse would otherwise choke on.
    args, _unknown = parser.parse_known_args()

    windows = [args.window] if args.window else list(range(1, 13))

    # Loop directly over FMin1.csv, FMin2.csv, FMin3.csv, ... -- no folder path.
    for w in windows:
        filename = f"FMin{w}.csv"
        if not os.path.exists(filename):
            print(f"Skipping window {w}: {filename} not found in this folder.")
            continue
        fr = pd.read_csv(filename, index_col=0)
        run_window(fr, window_label=filename)


if __name__ == "__main__":
    main()

C:\Users\Didar\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Didar\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (



Window: FMin1.csv

[mean (theta = 1/N, used in paper)]  theta = 0.02498
  mean out-degree: 22.26
  top 5 by influence: ['Trina_Solar', 'Eni', 'REW', 'ConocoPhillips', 'Williams']

[median]  theta = 0.02670
  mean out-degree: 19.49
  top 5 by influence: ['Trina_Solar', 'ConocoPhillips', 'Eni', 'REW', 'Williams']

[60th percentile]  theta = 0.02888
  mean out-degree: 15.59
  top 5 by influence: ['Trina_Solar', 'ConocoPhillips', 'REW', 'Tullow_Oil', 'Eni']

[75th percentile]  theta = 0.03199
  mean out-degree: 9.74
  top 5 by influence: ['Trina_Solar', 'Tullow_Oil', 'CNNC ', 'Williams', 'State_Grid']

Spearman rank correlation vs. 'mean (theta = 1/N, used in paper)':
  median              : rho = 0.987, p = 2.84e-31
  60th percentile     : rho = 0.950, p = 2.44e-20
  75th percentile     : rho = 0.892, p = 2.37e-14

Rank position (1 = most influential, out of 39) for key firms:
  mean (theta = 1/N, used in paper)  : {'BP': 13, 'Gazprom': 9, 'Inter_Rao': 13, 'State_Grid': 19, 'ExxonMobil':

In [3]:
"""
Random Forest importance stability check across random seeds
(Reviewer 2, Comment 2 -- sample adequacy / bootstrap stability).



Two stability measures are reported per window:

  [1] Row-wise importance stability: for each of the 39 companies,
      the Spearman correlation between its importance-of-predictors
      row at seed=42 and the same row at an alternative seed, averaged
      over all companies and all alternative seeds. Answers: "does
      each company get a similar RELATIVE ranking of which other
      companies matter to it, regardless of seed?"

  [2] Network stability: rebuilds the theta = 1/N network (same rule
      as your network-construction code) from each alternative-seed
      FR matrix, and reports the Spearman correlation of the resulting
      company out-degree ("influence count") ranking against the
      seed=42 ranking used in the paper. Answers: "does the final
      who-influences-whom conclusion change with the seed?"


"""

import argparse
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr


def build_fr_matrix(df, seed):
    """Reproduce the feature-ranking procedure exactly, for one seed."""
    feature_importance_scores = {}
    for feature_name in df.columns:
        y = df[feature_name].iloc[1:]   # dependent variable
        x = df.drop(df.index[-1])       # independent variables

        model = RandomForestRegressor(n_estimators=200, max_features='log2',
                                       random_state=seed)
        model.fit(x, y)
        score = model.feature_importances_

        # Invert the importance scores using (max - score)
        inverted = score.max() - score

        # Normalize to make them sum to 1
        if inverted.sum() > 0:
            inverted_normalized = inverted / inverted.sum()
        else:
            inverted_normalized = np.zeros_like(inverted)

        feature_importance_scores[feature_name] = inverted_normalized

    importance_matrix = np.array(list(feature_importance_scores.values()))
    feature_names = list(feature_importance_scores.keys())
    return pd.DataFrame(importance_matrix, index=feature_names, columns=df.columns)


def out_degree(fr_matrix):
    """Same rule as the network-construction code: A_ij = 1 if FR_ij > theta,
    theta = matrix-wide mean; out-degree = column sums (influence count)."""
    theta = fr_matrix.values.mean()
    A = (fr_matrix > theta).astype(int)
    return A.sum(axis=0)


def run_window(window_idx, n_seeds, base_seed=42):
    path = f"log_window_{window_idx}.csv"
    if not os.path.exists(path):
        print(f"Skipping window {window_idx}: {path} not found in this folder.")
        return None

    df = pd.read_csv(path, index_col=0)

    print(f"\n{'='*70}\nWindow {window_idx}: {path}\n{'='*70}")

    base_fr = build_fr_matrix(df, base_seed)
    base_degree = out_degree(base_fr)

    row_rhos = []
    degree_rhos = []

    alt_seeds = [s for s in range(1, n_seeds + 1) if s != base_seed]
    for seed in alt_seeds:
        alt_fr = build_fr_matrix(df, seed)

        # [1] row-wise stability: each company's own importance-of-predictors row
        for company in base_fr.index:
            rho, _ = spearmanr(base_fr.loc[company], alt_fr.loc[company])
            if not np.isnan(rho):
                row_rhos.append(rho)

        # [2] network-level stability: out-degree / influence-count ranking
        alt_degree = out_degree(alt_fr)
        rho, _ = spearmanr(base_degree, alt_degree)
        degree_rhos.append(rho)

    row_rhos = np.array(row_rhos)
    degree_rhos = np.array(degree_rhos)

    print(f"Alternative seeds tested (base seed = {base_seed}): {alt_seeds}")
    print(f"[1] Row-wise importance stability (per-company predictor ranking):")
    print(f"    mean rho = {row_rhos.mean():.3f}, sd = {row_rhos.std():.3f}, "
          f"min = {row_rhos.min():.3f}, n = {len(row_rhos)}")
    print(f"[2] Network out-degree (influence-count ranking) stability vs seed={base_seed}:")
    print(f"    mean rho = {degree_rhos.mean():.3f}, sd = {degree_rhos.std():.3f}, "
          f"min = {degree_rhos.min():.3f}")
    print(f"    per-seed rho: {[round(r, 3) for r in degree_rhos]}")

    return {"window": window_idx,
            "row_rho_mean": row_rhos.mean(), "row_rho_min": row_rhos.min(),
            "degree_rho_mean": degree_rhos.mean(), "degree_rho_min": degree_rhos.min()}


def main():
    parser = argparse.ArgumentParser(description="RF importance seed-stability check")
    parser.add_argument("--window", type=int, default=None,
                         help="Window number 1-12. Omit to run all windows.")
    parser.add_argument("--n_seeds", type=int, default=20,
                         help="Number of alternative random seeds to test "
                              "(seeds 1..n_seeds, excluding the paper's seed=42). Default 20.")
    # parse_known_args so this also runs cleanly inside Jupyter/IPython
    args, _unknown = parser.parse_known_args()

    windows = [args.window] if args.window else list(range(1, 13))

    summary = []
    for w in windows:
        result = run_window(w, args.n_seeds)
        if result:
            summary.append(result)

    if summary:
        print(f"\n{'='*70}\nOVERALL SUMMARY (across {len(summary)} windows)\n{'='*70}")
        s = pd.DataFrame(summary)
        print(s.to_string(index=False))
        print(f"\nAverage row-wise importance stability (all windows): {s['row_rho_mean'].mean():.3f}")
        print(f"Average network out-degree stability (all windows):   {s['degree_rho_mean'].mean():.3f}")
        s.to_csv("seed_stability_summary.csv", index=False)
        print("\nSaved per-window summary to seed_stability_summary.csv")


if __name__ == "__main__":
    main()


Window 1: log_window_1.csv


C:\Users\Didar\AppData\Local\Temp\ipykernel_4452\1293827292.py:117: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(base_fr.loc[company], alt_fr.loc[company])
C:\Users\Didar\AppData\Local\Temp\ipykernel_4452\1293827292.py:117: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(base_fr.loc[company], alt_fr.loc[company])
C:\Users\Didar\AppData\Local\Temp\ipykernel_4452\1293827292.py:117: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(base_fr.loc[company], alt_fr.loc[company])
C:\Users\Didar\AppData\Local\Temp\ipykernel_4452\1293827292.py:117: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(base_fr.loc[company], alt_fr.loc[company])
C:\Users\Didar\AppData\Local\Temp\ipykernel_4452\1293827292.py:117: ConstantInputWarning: An inp

Alternative seeds tested (base seed = 42): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
[1] Row-wise importance stability (per-company predictor ranking):
    mean rho = 0.808, sd = 0.075, min = 0.479, n = 760
[2] Network out-degree (influence-count ranking) stability vs seed=42:
    mean rho = 0.852, sd = 0.043, min = 0.740
    per-seed rho: [0.865, 0.881, 0.827, 0.822, 0.908, 0.874, 0.783, 0.901, 0.883, 0.819, 0.863, 0.892, 0.909, 0.856, 0.891, 0.81, 0.834, 0.854, 0.74, 0.83]

Window 2: log_window_2.csv
Alternative seeds tested (base seed = 42): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
[1] Row-wise importance stability (per-company predictor ranking):
    mean rho = 0.783, sd = 0.069, min = 0.530, n = 780
[2] Network out-degree (influence-count ranking) stability vs seed=42:
    mean rho = 0.829, sd = 0.040, min = 0.736
    per-seed rho: [0.828, 0.868, 0.81, 0.893, 0.849, 0.789, 0.829, 0.889, 0.823, 0.816, 0.889, 0.736, 0.831,

In [ ]:
"""
Case-resampling bootstrap stability check (Reviewer 2, Comment 2).



  [1] Network stability: Spearman correlation between the bootstrap
      replicate's out-degree (influence-count) ranking and the
      original (non-resampled) ranking used in the paper.

  [2] 95% confidence intervals on each company's out-degree, built
      directly from the bootstrap distribution across replicates --
      this is the literal "confidence interval" the reviewer asked
      about.

"""

import argparse
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr

RF_SEED = 42  # held fixed throughout -- isolates the resampling effect


def build_fr_matrix_from_index(df, idx):
    """idx: array of position indices into [0, len(df)-2], possibly with
    repeats (bootstrap draw). Reproduces the original feature-ranking
    procedure, but built from the resampled day-pairs (x_t, y_{t+1})
    instead of all N-1 pairs in original order."""
    x = df.iloc[idx].reset_index(drop=True)
    y_rows = [i + 1 for i in idx]

    feature_importance_scores = {}
    for feature_name in df.columns:
        y = df[feature_name].iloc[y_rows].reset_index(drop=True)

        model = RandomForestRegressor(n_estimators=200, max_features='log2',
                                       random_state=RF_SEED)
        model.fit(x, y)
        score = model.feature_importances_

        inverted = score.max() - score
        if inverted.sum() > 0:
            inverted_normalized = inverted / inverted.sum()
        else:
            inverted_normalized = np.zeros_like(inverted)

        feature_importance_scores[feature_name] = inverted_normalized

    importance_matrix = np.array(list(feature_importance_scores.values()))
    feature_names = list(feature_importance_scores.keys())
    return pd.DataFrame(importance_matrix, index=feature_names, columns=df.columns)


def out_degree(fr_matrix):
    theta = fr_matrix.values.mean()
    A = (fr_matrix > theta).astype(int)
    return A.sum(axis=0)


def run_window(window_idx, n_boot):
    path = f"log_window_{window_idx}.csv"
    if not os.path.exists(path):
        print(f"Skipping window {window_idx}: {path} not found in this folder.")
        return None, None

    df = pd.read_csv(path, index_col=0)
    n_pairs = len(df) - 1  # number of (x_t, y_{t+1}) day-pairs available

    print(f"\n{'='*70}\nWindow {window_idx}: {path}  ({n_pairs} day-pairs)\n{'='*70}")

    base_idx = list(range(n_pairs))          # original order, no resampling
    base_fr = build_fr_matrix_from_index(df, base_idx)
    base_degree = out_degree(base_fr)

    rng = np.random.RandomState(1000 + window_idx)
    degree_rhos = []
    boot_degrees = []  # list of Series, one per replicate

    for b in range(n_boot):
        boot_idx = rng.choice(n_pairs, size=n_pairs, replace=True)
        boot_fr = build_fr_matrix_from_index(df, boot_idx)
        boot_deg = out_degree(boot_fr)
        boot_degrees.append(boot_deg)

        rho, _ = spearmanr(base_degree, boot_deg)
        degree_rhos.append(rho)

    degree_rhos = np.array(degree_rhos)
    boot_df = pd.DataFrame(boot_degrees)  # rows = replicates, cols = companies

    ci_lower = boot_df.quantile(0.025)
    ci_upper = boot_df.quantile(0.975)

    print(f"Bootstrap replicates: {n_boot}")
    print(f"[1] Network out-degree stability under case-resampling bootstrap:")
    print(f"    mean rho = {degree_rhos.mean():.3f}, sd = {degree_rhos.std():.3f}, "
          f"min = {degree_rhos.min():.3f}")

    ci_table = pd.DataFrame({
        "baseline_out_degree": base_degree,
        "ci_2.5%": ci_lower,
        "ci_97.5%": ci_upper,
    })
    ci_table.index = [c.strip() for c in ci_table.index]

    print(f"[2] 95% bootstrap CI on out-degree, widest 8 intervals (company: baseline [lo, hi]):")
    ci_table["width"] = ci_table["ci_97.5%"] - ci_table["ci_2.5%"]
    widest = ci_table.sort_values("width", ascending=False).head(8)
    for name, row in widest.iterrows():
        print(f"    {name:<20} {row['baseline_out_degree']:.0f}  "
              f"[{row['ci_2.5%']:.1f}, {row['ci_97.5%']:.1f}]")

    return {"window": window_idx, "degree_rho_mean": degree_rhos.mean(),
            "degree_rho_min": degree_rhos.min()}, ci_table


def main():
    parser = argparse.ArgumentParser(description="Case-resampling bootstrap stability check")
    parser.add_argument("--window", type=int, default=None,
                         help="Window number 1-12. Omit to run all windows.")
    parser.add_argument("--n_boot", type=int, default=200,
                         help="Number of bootstrap replicates. Default 20.")
    args, _unknown = parser.parse_known_args()

    windows = [args.window] if args.window else list(range(1, 13))

    summary = []
    all_ci = []
    for w in windows:
        result, ci_table = run_window(w, args.n_boot)
        if result:
            summary.append(result)
            ci_table["window"] = w
            all_ci.append(ci_table)

    if summary:
        print(f"\n{'='*70}\nOVERALL SUMMARY (across {len(summary)} windows)\n{'='*70}")
        s = pd.DataFrame(summary)
        print(s.to_string(index=False))
        print(f"\nAverage network out-degree bootstrap stability: {s['degree_rho_mean'].mean():.3f}")
        s.to_csv("bootstrap_stability_summary.csv", index=False)
        pd.concat(all_ci).to_csv("bootstrap_confidence_intervals.csv")
        print("Saved bootstrap_stability_summary.csv and bootstrap_confidence_intervals.csv")


if __name__ == "__main__":
    main()


Window 1: log_window_1.csv  (102 day-pairs)
